# OpsMind: QLoRA-дообучение классификатора тикетов

Задача: по теме и описанию обращения вернуть JSON `{category, priority, team}`.

**Перед запуском в Kaggle:**
1. Settings → Accelerator → **GPU T4 x2** (или P100). Internet → **On** (нужна верификация телефона в аккаунте Kaggle).
2. Add Input → Upload → новый Dataset `opsmind-finetune` из файлов `data/finetune/train.jsonl`, `val.jsonl`, `test.jsonl`, `test_hard.jsonl`.
3. Run All. Обучение ~15 минут, экспорт GGUF ещё ~10.
4. Save Version → в Output появятся `opsmind-lora.zip` и `opsmind-gguf/*.gguf` — скачать.

Что происходит:
- базовая модель загружается в 4 битах (**Q** в QLoRA), к слоям внимания и MLP добавляются LoRA-адаптеры (~1% параметров), обучаются только они;
- loss считается только по ответу ассистента (`train_on_responses_only`) — модель учится отвечать, а не переписывать промпт;
- качество меряем **до** и **после** на `test` (шаблонные формулировки) и `test_hard` (написан вручную, честная проверка).

In [ ]:
%%capture
!pip install -q unsloth
# Если установка или импорт падают — версии Unsloth/TRL меняются часто: см. https://docs.unsloth.ai (Kaggle notebooks)

In [ ]:
import glob, json, os, re, time

MODEL_NAME = "unsloth/Qwen3-1.7B"   # альтернатива покрупнее: "unsloth/Qwen3-4B-Instruct-2507"
MAX_SEQ_LEN = 512                    # тикет + системный промпт + ответ укладываются в ~300 токенов
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
print("data:", DATA_DIR, os.listdir(DATA_DIR))

def read_jsonl(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_rows, val_rows, test_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl"), read_jsonl("test.jsonl")
SYSTEM = train_rows[0]["messages"][0]["content"]

# test_hard написан в простом формате — приводим к тому же виду, что и обучающие примеры
hard_rows = [{
    "id": r["id"],
    "messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Тема: {r['subject']}\nОписание: {r['description']}"},
        {"role": "assistant", "content": json.dumps({"category": r["category"], "priority": r["priority"], "team": r["team"]}, ensure_ascii=False)},
    ]} for r in read_jsonl("test_hard.jsonl")]
print(len(train_rows), len(val_rows), len(test_rows), len(hard_rows))
print(SYSTEM)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,        # QLoRA: базовые веса в 4 битах, адаптеры — в 16
)
print(f"Параметров в модели: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")

In [ ]:
# Оценка: та же логика, что в tools/OpsMind.ClassifierEval (C#), чтобы цифры были сравнимы
def classify(user_text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_text}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=60, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def parse(raw):
    m = re.search(r"\{[^{}]*\}", raw)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def evaluate(rows, name):
    FastLanguageModel.for_inference(model)
    stats = {"valid": 0, "category": 0, "priority": 0, "team": 0, "exact": 0}
    errors, started = [], time.time()
    for r in rows:
        truth = json.loads(r["messages"][2]["content"])
        raw = classify(r["messages"][1]["content"])
        pred = parse(raw) or {}
        stats["valid"] += bool(pred)
        ok = {k: str(pred.get(k, "")).strip().lower() == truth[k].lower() for k in ("category", "priority", "team")}
        for k, v in ok.items():
            stats[k] += v
        stats["exact"] += all(ok.values())
        if not all(ok.values()) and len(errors) < 8:
            errors.append((r["id"], truth, raw.strip()[:120]))
    n = len(rows)
    print(f"{name} ({n}): " + ", ".join(f"{k} {v / n:.0%}" for k, v in stats.items()) + f", {(time.time() - started) / n:.2f} с/тикет")
    for e in errors:
        print("   ", e)
    return {k: v / n for k, v in stats.items()}

baseline = {"test": evaluate(test_rows[:150], "ДО обучения, test"), "hard": evaluate(hard_rows, "ДО обучения, hard")}

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # ранг адаптера: размер «поправки». 8–32 — обычный диапазон
    lora_alpha=16,           # масштаб поправки (часто alpha = r)
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

# Каждый пример — диалог system/user/assistant в chat-шаблоне модели.
# enable_thinking=False: Qwen3 не должен «рассуждать», ответ — сразу JSON (так же модель будет вызываться в проде).
def to_text(rows):
    return Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize=False, enable_thinking=False)} for r in rows])

train_ds, val_ds = to_text(train_rows), to_text(val_rows)
print(train_ds[0]["text"])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,   # эффективный батч 16
        num_train_epochs=2,              # 2400 примеров × 2 эпохи ≈ 300 шагов
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,                   # следим за val loss: если растёт, а train падает — переобучение
        per_device_eval_batch_size=8,
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

# Loss только по ответу ассистента: системный промпт и тикет модель не должна «учиться писать»
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

FastLanguageModel.for_training(model)
stats = trainer.train()
print(stats)

In [ ]:
after = {"test": evaluate(test_rows, "ПОСЛЕ обучения, test"), "hard": evaluate(hard_rows, "ПОСЛЕ обучения, hard")}

print("\n| Набор | Метрика | До | После |\n|---|---|---|---|")
for s in ("test", "hard"):
    for k in ("valid", "category", "priority", "team", "exact"):
        print(f"| {s} | {k} | {baseline[s][k]:.0%} | {after[s][k]:.0%} |")

In [ ]:
# 1) LoRA-адаптер (~десятки МБ): для vLLM --enable-lora поверх базовой модели
model.save_pretrained("opsmind-lora")
tokenizer.save_pretrained("opsmind-lora")
!cd /kaggle/working && zip -qr opsmind-lora.zip opsmind-lora && ls -lh opsmind-lora.zip

In [ ]:
# 2) GGUF q4_k_m (~1 ГБ): адаптер вливается в модель и квантуется для llama.cpp / Ollama на CPU
model.save_pretrained_gguf("opsmind-gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh /kaggle/working/opsmind-gguf